# Viande et masculinité
## Existe-t-il une corrélation entre les normes de genre et la consommation de viande rouge en Europe ?

**Projet de data science — ENSAE Paris, 2A**

Autrices: Solène Pinguet - Noriane Ménard - Emna Larbi · *Date : 24/11/2026*

---

### Résumé
> *À rédiger en dernier (4-5 lignes) : question, données, méthode, résultat principal, limite principale.*

### Plan
0. [Configuration](#1.-Configuration)
1. [Les données](#2.-Les-données)
2. [Contrôles de qualité](#3.-Contrôles-de-qualité)
3. [Statistiques descriptives](#4.-Statistiques-descriptives)
4. [Cartographie](#5.-Cartographie)
5. [Analyse bivariée](#6.-Analyse-bivariée)
6. [Régressions](#7.-Régressions)
7. [Scraping de publicités](#8.-Scraping-de-publicités)
8. [Discussion et conclusion](#9.-Discussion-et-conclusion)
9. [Annexe : reproductibilité](#10.-Annexe-:-reproductibilité)

## 1. Configuration

Le notebook **affiche et commente** ; les calculs sont faits par les scripts de `Src/`, importés ci-dessous.
Avec `RECALCULER = False`, le notebook lit les fichiers déjà produits dans `Data/processed/` (exécution rapide).
Avec `RECALCULER = True`, il relance la chaîne complète depuis `Data/raw/`.

In [5]:
import subprocess
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col
from statsmodels.stats.outliers_influence import variance_inflation_factor
from IPython.display import Image, Markdown, display

RACINE = Path.cwd()
if not (RACINE / "Src").exists():
    RACINE = next(p for p in [RACINE, *RACINE.parents] if (p / "Src").exists())
SRC = RACINE / "Src"
RAW = RACINE / "Data" / "raw"
PROCESSED = RACINE / "Data" / "processed"
sys.path.insert(0, str(SRC))

RECALCULER = False          # True : relance le nettoyage, la jointure et les cartes
GRAINE = 2020
np.random.seed(GRAINE)

pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "font.family": "serif"})
warnings.filterwarnings("ignore", category=FutureWarning)

print("Racine du projet :", RACINE)

Racine du projet : /home/onyxia/Projet_01_ENSAE_2A


In [6]:
# Chaîne de traitement complète (uniquement si RECALCULER = True).
# Adapter la liste au nom réel des scripts de Src/.
ETAPES = ["clean.py", "maps.py", "carte_masculinite.py"]

if RECALCULER:
    for script in ETAPES:
        print(f"▶ {script}")
        subprocess.run([sys.executable, str(SRC / script)], cwd=RACINE, check=True)
else:
    print("Lecture des fichiers déjà produits dans Data/processed/")

Lecture des fichiers déjà produits dans Data/processed/


## 2. Les données

### 2.1 Sources

| Source | Années | Contenu utilisé | Unité | Pondération |
|---|---|---|---|---|
| ISSP 2020 — *Environment IV* | 2020-2022 | fréquence de consommation de viande rouge | individu | `poids` |
| ESS round 11 | 2023-2024 | masculinité ressentie (`mascfel`), attitudes | individu | 0-6|
| PIB par habitant | *année à préciser* | niveau de richesse | pays |-|


### 2.2 Variables

| Variable | Source | Question / définition | Échelle |
|---|---|---|---|
| `jours_viande_rouge` | ISSP | fréquence déclarée de consommation de viande rouge | *à préciser* |
| `ecart_h_f` | ISSP (calculée) | moyenne pondérée hommes − moyenne pondérée femmes | - |
| `mascfel` | ESS | « À quel point vous sentez-vous masculin ? » | 0-6 |
| `mascfel_homme`, `mascfel_femme` | ESS (calculées) | moyenne par pays et par sexe | 0-6 |
| `ecart_masc_h_f` | calculée | `mascfel_homme − mascfel_femme` | - |
| `pib_hab` | PIB | PIB par habitant, utilisé en logarithme |-|

### 2.3 Nettoyage


Le nettoyage est réalisé par `Src/clean.py`, qui traite séparément les deux enquêtes puis les agrège au niveau des pays.
Les fichiers produits sont enregistrés dans `Data/processed/`.

#### ISSP 2020 : consommation de viande rouge

**Variables retenues.** Seules cinq variables sont chargées depuis `ISSP_2020.dta` :

| Variable brute | Renommée en | Contenu |
|---|---|---|
| `c_alphan` | `country_iso2` | code pays ISO à deux lettres |
| `country` | `country_code` | code pays numérique ISSP |
| `SEX` | `sexe` | sexe du répondant |
| `v50` | `jours_viande_rouge` | nombre de jours de consommation de viande rouge (fréquence déclarée, pas une quantité) |
| `WEIGHT` | `poids` | poids statistique individuel |

**Codes de non-réponse.** Dans l'ISSP, les codes négatifs ne sont pas des réponses :
- les répondants qui déclarent **ne pas manger de viande** (code −4) sont recodés à **0 jour**. Ils ne consomment
  réellement aucune viande rouge, et les exclure aurait surestimé la consommation moyenne. Ce choix compte d'autant plus que
  les femmes sont plus souvent végétariennes : les retirer aurait mécaniquement réduit l'écart entre hommes et femmes ;
- tous les autres codes négatifs (« ne sait pas », refus, non-réponse) sont traités comme **valeurs manquantes** ;
- le sexe est recodé `1 → homme`, `2 → femme` ; toute autre valeur est considérée comme manquante.

Les répondants sans sexe, sans fréquence de consommation ou sans poids sont ensuite retirés,
car ils ne peuvent pas entrer dans le calcul des moyennes par sexe.

**Pondération.** Les échantillons ISSP ne sont pas parfaitement représentatifs de la population
(certaines catégories sont sur- ou sous-représentées). Toutes les moyennes sont donc **pondérées** par `WEIGHT` :

$$\bar{x}_{\text{pays, sexe}} = \frac{\sum_i w_i\, x_i}{\sum_i w_i}$$

**Agrégation par pays et par sexe.** Pour chaque pays, on calcule :
- `conso_homme` et `conso_femme` : moyennes pondérées de la fréquence de consommation ;
- `n_homme` et `n_femme` : effectifs **bruts** (non pondérés), qui servent à repérer les petits échantillons ;
- `ecart_h_f = conso_homme − conso_femme` : positif quand les hommes déclarent une fréquence plus élevée.

#### ESS 11 : masculinité ressentie et attitudes

**Variables retenues.** Le code pays (`cntry`), le sexe (`gndr`), la masculinité ressentie (`mascfel`),
l'importance d'être un homme ou une femme dans l'identité (`impbemw`) et des variables d'attitudes
(climat, politique, médias, alcool).

**Codes de non-réponse.** Chaque variable ESS a ses propres codes de refus, de « ne sait pas » et de non-réponse
(par exemple 7, 8, 9 pour une échelle de 0 à 6 ; 77, 88, 99 pour une échelle de 0 à 10). Ces codes sont remplacés
par des valeurs manquantes, variable par variable. Une variable `climatosceptique` est créée à partir de `ccnthum`
(1 si le répondant déclare ne pas croire au changement climatique, code 55).

**Lignes incomplètes.** Les répondants ayant une valeur manquante sur au moins une variable sont retirés
(suppression des cas incomplets).

**Agrégation par pays et par sexe.** Pour chaque pays, on calcule la moyenne de chaque variable sur l'ensemble des
répondants, puis `mascfel_homme` et `mascfel_femme`, moyennes de la masculinité ressentie chez les hommes et chez les
femmes, avec leurs effectifs (`n_ess_homme`, `n_ess_femme`). L'agrégation par sexe permet de construire
`ecart_masc_h_f = mascfel_homme − mascfel_femme`, qui se compare directement à l'écart de consommation de viande.


### 2.4 Jointure 
La table d'analyse `Data/processed/table_jointe.csv` est construite par `Src/jointure.py`. Elle réunit,
**pour chaque pays**, trois sources :

| Source | Fichier | Clé pays | Contenu apporté |
|---|---|---|---|
| ESS 11 | `ess11_agrege_pays.csv` | ISO2 (`code_pays`) | masculinité ressentie, attitudes |
| ISSP 2020 | `issp_par_pays.csv` | ISO2 (`country_iso2`) | fréquences de viande rouge par sexe, écart H − F |
| Banque mondiale | `gdp_world_countries_percapita.csv` | ISO3 (`Country Code`) | PIB par habitant |

L'unité d'observation est le **pays** : l'ISSP et l'ESS interrogent des personnes différentes, à des dates
différentes. On ne peut donc rapprocher que des moyennes nationales, pas des individus.

#### Étape 1 : harmonisation des clés

Les colonnes pays des deux enquêtes sont renommées en `iso2` pour servir de clé commune.

#### Étape 2 : contrôles avant jointure

- **Unicité** : on vérifie qu'aucun pays n'apparaît deux fois dans l'ESS ou dans l'ISSP. Un doublon dupliquerait
  des lignes lors de la jointure et fausserait toutes les analyses.
- **Couverture** : on affiche les pays présents dans une seule des deux enquêtes, pour savoir lesquels seront perdus.

#### Étape 3 : jointure ESS × ISSP

Jointure **interne** (`how="inner"`) sur `iso2` : seuls les pays présents **dans les deux enquêtes** sont conservés,
car l'analyse demande à la fois la masculinité ressentie (ESS) et la consommation de viande (ISSP).
L'option `validate="one_to_one"` fait échouer le script si un pays correspondait à plusieurs lignes.

Pays perdus à cette étape :
- présents seulement dans l'ESS : [à compléter] ;
- présents seulement dans l'ISSP : [à compléter, par exemple les pays hors d'Europe].

#### Étape 4 : ajout du PIB par habitant

La Banque mondiale identifie les pays par leur code **ISO3**. On convertit donc `iso2` en `iso3` avec le paquet
`country_converter`, qui applique la norme ISO 3166, puis on fait une jointure **à gauche** (`how="left"`) :
tous les pays de l'étape 3 sont conservés, et un pays absent de la table du PIB aurait une valeur manquante plutôt
que d'être supprimé sans avertissement.

**Année retenue : 2022.** Elle se situe entre les terrains de l'ISSP (2020-2022) et de l'ESS 11 (2023-2024).
Le PIB par habitant évoluant lentement, le choix d'une année voisine changerait peu le classement des pays.

#### Résultat

| | |
|---|---|
| Pays retenus | 15 |
| Colonnes | 24 |
| Valeurs manquantes | aucune dans les variables d'analyse |

Pays retenus : [à compléter avec la liste des codes ISO2].

#### Points de vigilance

- **Codes pays non harmonisés.** Une jointure interne écarte sans le dire un pays codé différemment dans les deux
  sources (par exemple `GB` / `UK` ou `GR` / `EL`). L'affichage des pays présents dans une seule enquête
  (étape 2) permet de vérifier qu'aucun pays n'a été perdu pour cette raison.
- **Conversion ISO3.** Pour un code qu'il ne reconnaît pas, `country_converter` ne renvoie pas une valeur
  manquante mais le texte `"not found"`. Ce pays n'obtient alors pas de PIB, ce que le contrôle final des
  valeurs manquantes permet de détecter.
- **Unité du PIB.** [À préciser : dollars courants ou parité de pouvoir d'achat (PPA)]. Le PIB en PPA corrige les
  différences de prix entre pays et mesure mieux le niveau de vie réel.

La table d'analyse `Data/processed/table_jointe.csv` est construite par `Src/jointure.py`. Elle réunit,
**pour chaque pays**, trois sources :

| Source | Fichier | Clé pays | Contenu apporté |
|---|---|---|---|
| ESS 11 | `ess11_agrege_pays.csv` | ISO2 (`code_pays`) | masculinité ressentie, attitudes |
| ISSP 2020 | `issp_par_pays.csv` | ISO2 (`country_iso2`) | fréquences de viande rouge par sexe, écart H − F |
| Banque mondiale | `gdp_world_countries_percapita.csv` | ISO3 (`Country Code`) | PIB par habitant |

L'unité d'observation est le **pays** : l'ISSP et l'ESS interrogent des personnes différentes, à des dates
différentes. On ne peut donc rapprocher que des moyennes nationales, pas des individus.

#### Étape 1 : harmonisation des clés

Les colonnes pays des deux enquêtes sont renommées en `iso2` pour servir de clé commune.

#### Étape 2 : contrôles avant jointure

- **Unicité** : on vérifie qu'aucun pays n'apparaît deux fois dans l'ESS ou dans l'ISSP. Un doublon dupliquerait
  des lignes lors de la jointure et fausserait toutes les analyses.
- **Couverture** : on affiche les pays présents dans une seule des deux enquêtes, pour savoir lesquels seront perdus.

#### Étape 3 : jointure ESS × ISSP

Jointure **interne** (`how="inner"`) sur `iso2` : seuls les pays présents **dans les deux enquêtes** sont conservés,
car l'analyse demande à la fois la masculinité ressentie (ESS) et la consommation de viande (ISSP).
L'option `validate="one_to_one"` fait échouer le script si un pays correspondait à plusieurs lignes.

Pays perdus à cette étape :
- présents seulement dans l'ESS : [à compléter] ;
- présents seulement dans l'ISSP : [à compléter, par exemple les pays hors d'Europe].

#### Étape 4 : ajout du PIB par habitant

La Banque mondiale identifie les pays par leur code **ISO3**. On convertit donc `iso2` en `iso3` avec le paquet
`country_converter`, qui applique la norme ISO 3166, puis on fait une jointure **à gauche** (`how="left"`) :
tous les pays de l'étape 3 sont conservés, et un pays absent de la table du PIB aurait une valeur manquante plutôt
que d'être supprimé sans avertissement.

**Année retenue : 2022.** Elle se situe entre les terrains de l'ISSP (2020-2022) et de l'ESS 11 (2023-2024).
Le PIB par habitant évoluant lentement, le choix d'une année voisine changerait peu le classement des pays.

#### Résultat

| | |
|---|---|
| Pays retenus | 15 |
| Colonnes | 24 |
| Valeurs manquantes | aucune dans les variables d'analyse |

Pays retenus : [à compléter avec la liste des codes ISO2].

#### Points de vigilance

- **Codes pays non harmonisés.** Une jointure interne écarte sans le dire un pays codé différemment dans les deux
  sources (par exemple `GB` / `UK` ou `GR` / `EL`). L'affichage des pays présents dans une seule enquête
  (étape 2) permet de vérifier qu'aucun pays n'a été perdu pour cette raison.
- **Conversion ISO3.** Pour un code qu'il ne reconnaît pas, `country_converter` ne renvoie pas une valeur
  manquante mais le texte `"not found"`. Ce pays n'obtient alors pas de PIB, ce que le contrôle final des
  valeurs manquantes permet de détecter.
- **Unité du PIB.** [À préciser : dollars courants ou parité de pouvoir d'achat (PPA)]. Le PIB en PPA corrige les
  différences de prix entre pays et mesure mieux le niveau de vie réel.

## 3. Contrôles de qualité

Vérifications avant toute analyse : valeurs manquantes, doublons de pays, effectifs par sexe,
cohérence entre le recalcul pondéré et `issp_par_pays.csv`.

In [7]:
print("Valeurs manquantes dans table_jointe :")
display(table.isna().sum().to_frame("NA").query("NA > 0"))
print("Doublons de pays :", table["iso2"].duplicated().sum())

Valeurs manquantes dans table_jointe :


NameError: name 'table' is not defined

In [ ]:
# Effectifs par sexe : signaler les petits échantillons
SEUIL = 100
eff = table[["iso2", "n_homme", "n_femme"]].copy()
if {"n_ess_homme", "n_ess_femme"} <= set(table.columns):
    eff[["n_ess_homme", "n_ess_femme"]] = table[["n_ess_homme", "n_ess_femme"]]
faibles = eff[(eff.drop(columns="iso2") < SEUIL).any(axis=1)]
print(f"Pays avec moins de {SEUIL} répondants pour un sexe :",
      faibles["iso2"].tolist() or "aucun")
display(eff.sort_values("n_homme"))

In [ ]:
# Contrôle croisé ISSP : recalcul pondéré (Src/maps.py) contre issp_par_pays.csv
individus = maps.charger_individuel()
recalc = maps.agreger_par_pays(individus)
maps.controler_fichier_agrege(recalc)

## 4. Statistiques descriptives

In [ ]:
desc = d[VARS].agg(["count", "mean", "std", "min", "max"]).T
desc.columns = ["N", "Moyenne", "Écart-type", "Min", "Max"]
desc

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
titres = {"ecart_viande_h_f": "Écart viande rouge H − F",
          "ecart_masc_h_f": "Écart masculinité ressentie H − F",
          "log_pib": "log(PIB par habitant)"}
for ax, v in zip(axes, VARS):
    ax.hist(d[v], bins=8, color="#5fa596", edgecolor="white")
    ax.set_title(titres[v], fontsize=11)
    ax.set_ylabel("Nombre de pays")
plt.tight_layout()
plt.show()

*Commentaire : …*

## 5. Cartographie

Les cartes interactives (infobulles, menu des vues) sont dans `Data/processed/*.html`.
GitHub n'exécute pas le JavaScript des notebooks : on affiche donc ici les **images PNG**.

### 5.1 Écart de consommation de viande rouge entre hommes et femmes (ISSP 2020)

In [ ]:
# Image de la carte de l'écart de viande (Src/maps.py ne produit que le HTML)
png_viande = PROCESSED / "carte_viande_genre_ecart.png"
if RECALCULER or not png_viande.exists():
    geo = maps.ajouter_geographie(recalc)
    fig = maps.construire_figure(geo)
    fig.layout.updatemenus = ()                       # pas de menu sur une image fixe
    fig.update_layout(width=1100, height=750,
                      title="Écart de fréquence de consommation de viande rouge H − F — ISSP 2020")
    try:
        fig.write_image(png_viande, scale=2)          # nécessite kaleido + Internet
    except Exception as e:
        print("Export PNG impossible :", e)
if png_viande.exists():
    display(Image(png_viande, width=850))
print("Version interactive :", PROCESSED / "carte_viande_genre.html")

### 5.2 Masculinité ressentie (ESS 11)

In [ ]:
for vue in ["mascfel_homme", "mascfel_femme", "ecart_mascfel"]:
    chemin = PROCESSED / f"carte_masculinite_{vue}.png"
    if chemin.exists():
        display(Image(chemin, width=850))
    else:
        print("Image absente — lancer : python Src/carte_masculinite.py →", chemin.name)
print("Version interactive :", PROCESSED / "carte_masculinite.html")

### 5.3 Lecture comparée

*Commentaire : les pays où l'écart de viande est fort ont-ils aussi des identités de genre plus différenciées ?*

## 6. Analyse bivariée

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(d["ecart_masc_h_f"], d["ecart_viande_h_f"], s=45, color="#1f6f5c", zorder=3)
for _, r in d.iterrows():
    ax.annotate(r["iso2"], (r["ecart_masc_h_f"], r["ecart_viande_h_f"]),
                textcoords="offset points", xytext=(4, 4), fontsize=9, color="#52514e")
pente, origine = np.polyfit(d["ecart_masc_h_f"], d["ecart_viande_h_f"], 1)
xs = np.linspace(d["ecart_masc_h_f"].min(), d["ecart_masc_h_f"].max(), 50)
ax.plot(xs, origine + pente * xs, ls=":", color="#9b9a95")
ax.set_xlabel("Écart de masculinité ressentie H − F (ESS 11)")
ax.set_ylabel("Écart de fréquence de viande rouge H − F (ISSP 2020)")
ax.grid(alpha=.3)
plt.show()

In [ ]:
print("Pearson :")
display(d[VARS].corr(method="pearson"))
print("Spearman :")
display(d[VARS].corr(method="spearman"))

*Commentaire : …*

## 7. Régressions

### 7.1 Spécification

Au niveau des pays $c$ :

$$\text{écart viande}_c = \alpha + \beta\,\text{écart masculinité}_c + \gamma \log(\text{PIB/hab}_c) + \varepsilon_c$$

- $\beta$ : association entre écart de masculinité et écart de viande **à PIB par habitant égal**.
- PIB en **logarithme** : distribution asymétrique et effet décroissant de la richesse.
- Erreurs-types robustes **HC3** et loi de **Student** (`use_t=True`), adaptées à un petit échantillon.

### 7.2 Résultats

In [ ]:
AJUST = dict(cov_type="HC3", use_t=True)
m1 = smf.ols("ecart_viande_h_f ~ ecart_masc_h_f", data=d).fit(**AJUST)
m2 = smf.ols("ecart_viande_h_f ~ ecart_masc_h_f + log_pib", data=d).fit(**AJUST)

def tableau(modeles, noms):
    return summary_col(modeles, model_names=noms, stars=True, float_format="%.3f",
                       info_dict={"N": lambda m: f"{int(m.nobs)}"},
                       regressor_order=["ecart_masc_h_f", "log_pib", "Intercept"])

tableau([m1, m2], ["(1) sans contrôle", "(2) avec PIB"])

In [ ]:
m2.summary()

### 7.3 Lecture du contrôle par le PIB

*Commentaire : évolution de $\beta$ entre (1) et (2), rôle de la richesse.*

### 7.4 Diagnostics

In [ ]:
# Colinéarité
X = sm.add_constant(d[["ecart_masc_h_f", "log_pib"]])
vif = pd.Series([variance_inflation_factor(X.values, i) for i in range(1, X.shape[1])],
                index=X.columns[1:], name="VIF")
display(vif.round(2))

# Pays influents
d["cook"] = m2.get_influence().cooks_distance[0]
influents = d.loc[d["cook"] > 4 / len(d), ["iso2", "cook"]]
print("Pays influents (Cook > 4/N) :")
display(influents)

In [ ]:
# Résidus
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].scatter(m2.fittedvalues, m2.resid, color="#1f6f5c")
axes[0].axhline(0, color="#9b9a95", ls=":")
axes[0].set(xlabel="Valeurs ajustées", ylabel="Résidus", title="Résidus contre valeurs ajustées")
sm.qqplot(m2.resid, line="s", ax=axes[1])
axes[1].set_title("QQ-plot des résidus")
plt.tight_layout()
plt.show()

### 7.5 Robustesse

In [ ]:
spec = "ecart_viande_h_f ~ ecart_masc_h_f + log_pib"
sans_hu = smf.ols(spec, data=d[d.iso2 != "HU"]).fit(**AJUST)
sans_ru = smf.ols(spec, data=d[d.iso2 != "RU"]).fit(**AJUST)
sans_influents = smf.ols(spec, data=d[~d.iso2.isin(influents.iso2)]).fit(**AJUST)

# Spécification alternative : niveaux chez les hommes plutôt qu'écarts
hommes = smf.ols("conso_homme ~ mascfel_homme + log_pib", data=d).fit(**AJUST)

tableau([m2, sans_hu, sans_ru, sans_influents],
        ["(2) référence", "sans HU", "sans RU", "sans pays influents"])

In [ ]:
summary_col([hommes], model_names=["Hommes : conso ~ mascfel"], stars=True,
            float_format="%.3f", info_dict={"N": lambda m: f"{int(m.nobs)}"})

### 7.6 Interprétation

*À rédiger : signe et ampleur de $\beta$, significativité (au seuil de 10 % au mieux avec 15 pays),
stabilité selon les tests de robustesse, pistes d'explication, limites
(association entre pays, pas de causalité, pas de conclusion individuelle).*

## 8. Scraping de publicités

*Partie à compléter.*

### 8.1 Objectif

*À compléter.*

### 8.2 Source et cadre

*À compléter.*

### 8.3 Collecte

*À compléter.*

### 8.4 Nettoyage

*À compléter.*

### 8.5 Analyse

*À compléter.*

### 8.6 Résultats et limites

*À compléter.*

## 9. Discussion et conclusion

### 9.1 Synthèse des résultats
*…*

### 9.2 Limites
- **Niveau d'analyse** : association entre pays ; aucune conclusion sur les individus (sophisme écologique).
- **Taille d'échantillon** : 15 pays, puissance statistique faible.
- **Mesure** : `mascfel` mesure une identité ressentie, indicateur approché des normes de genre.
- **Données déclaratives** et enquêtes menées à des dates différentes (ISSP 2020, ESS 11).
- **Causalité** : aucune relation causale ne peut être établie.

### 9.3 Pistes
*…*

## 10. Annexe : reproductibilité

Ordre d'exécution depuis la racine du projet :

```bash
pip install -r requirements.txt
python Src/clean.py
python Src/maps.py
python Src/carte_masculinite.py
python Src/regression.py
```

In [ ]:
import importlib.metadata as md_
paquets = ["pandas", "numpy", "statsmodels", "plotly", "pycountry", "matplotlib", "kaleido"]
versions = {}
for p in paquets:
    try:
        versions[p] = md_.version(p)
    except md_.PackageNotFoundError:
        versions[p] = "non installé"
print("Python", sys.version.split()[0])
display(pd.Series(versions, name="version").to_frame())

In [ ]:
print("Fichiers produits dans Data/processed/ :")
for f in sorted(PROCESSED.iterdir()):
    if f.is_file():
        print(f"  {f.name:<45} {f.stat().st_size / 1e6:6.2f} Mo")